# 📊 Documentación del Modelo Random Forest — CASSERISISSIMA 2.0

**Trabajo Especial de Grado**  
Universidad de Oriente — Núcleo de Monagas  
Autores: Br. Jorfran Gil · Br. Yefferson Hernández  

---

Este notebook documenta en detalle el modelo de Machine Learning utilizado en CASSERISISSIMA 2.0 para la predicción de demanda de productos de repostería artesanal. Sirve como referencia técnica para la tesis y para cualquier persona que quiera entender cómo funciona el motor predictivo.

## 1. Introducción al Problema

### 1.1 Contexto del Negocio

La Pastelería Casseríssimas produce tortas artesanales con una **vida útil de 3 a 4 días**. Esto genera un problema clásico de la gestión de inventarios perecederos:

- **Sobreproducción** → Merma económica (productos que se vencen y se descartan)
- **Subproducción** → Ventas perdidas (clientes que no encuentran el producto)

### 1.2 ¿Por qué Random Forest?

Se eligió **Random Forest Regressor** (Breiman, 2001) como modelo principal por las siguientes razones:

| Característica | Ventaja para este caso |
|----------------|------------------------|
| **Robusto a outliers** | Los pedidos de eventos especiales no distorsionan el modelo |
| **No requiere escalado** | Las features de calendario y lags tienen escalas muy diferentes |
| **Intervalos de confianza nativos** | Cada árbol da una predicción, lo que permite calcular bandas de incertidumbre |
| **Maneja bien datos faltantes** | Con `SimpleImputer` en el pipeline |
| **Interpretable** | Feature Importance permite entender qué impulsa la demanda |

Adicionalmente, se implementó **LightGBM** como competidor. El sistema entrena ambos modelos y selecciona automáticamente el ganador por RMSE.

## 2. Carga y Exploración de Datos

Los datos de ventas fueron proporcionados por la Pastelería Casseríssimas a través de archivos CSV y cargados en una base de datos SQLite.

In [ ]:
import sys
import os

# Agregar src/ al path para importar módulos del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos para la tesis
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print(f'Project root: {PROJECT_ROOT}')
print(f'Source directory: {SRC_DIR}')

In [ ]:
from db.database import SessionLocal, init_db
from db.models import Product, SaleTransaction

# Inicializar la base de datos
init_db()
db = SessionLocal()

# Cargar productos activos
products = db.query(Product).filter(Product.is_active == True).all()
print(f'Productos activos: {len(products)}')
print()

# Mostrar catálogo
catalog = pd.DataFrame([{
    'SKU': p.sku,
    'Nombre': p.name,
    'Categoría': p.category,
    'Precio Venta': f'${p.selling_price:.2f}',
    'Costo': f'${p.unit_cost:.2f}',
    'Vida Útil (días)': p.shelf_life_days,
} for p in products])

catalog

In [ ]:
# Cargar ventas del Escenario 2 (Óptimo - 2 años de historia)
SCENARIO_ID = 2

sales_rows = (
    db.query(
        SaleTransaction.product_id,
        SaleTransaction.sale_date,
        SaleTransaction.quantity_sold,
        SaleTransaction.revenue,
    )
    .filter(SaleTransaction.scenario_id == SCENARIO_ID)
    .order_by(SaleTransaction.sale_date)
    .all()
)

sales_df = pd.DataFrame(sales_rows, columns=['product_id', 'sale_date', 'quantity_sold', 'revenue'])
sales_df['sale_date'] = pd.to_datetime(sales_df['sale_date'])

print(f'Total de registros de ventas (Escenario {SCENARIO_ID}): {len(sales_df):,}')
print(f'Período: {sales_df.sale_date.min().date()} a {sales_df.sale_date.max().date()}')
print(f'Días de historia: {(sales_df.sale_date.max() - sales_df.sale_date.min()).days}')

In [ ]:
# Seleccionar un producto representativo para análisis detallado
PRODUCT_SKU = products[0].sku  # Primer producto del catálogo
product = [p for p in products if p.sku == PRODUCT_SKU][0]

product_sales = sales_df[sales_df['product_id'] == product.id].copy()
product_sales = product_sales.sort_values('sale_date').reset_index(drop=True)

print(f'Producto seleccionado: {product.name} ({product.sku})')
print(f'Registros de venta: {len(product_sales)}')
print(f'Venta promedio diaria: {product_sales.quantity_sold.mean():.2f} unidades')
print(f'Desviación estándar: {product_sales.quantity_sold.std():.2f}')

# Gráfico de serie temporal
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(product_sales['sale_date'], product_sales['quantity_sold'], 
        linewidth=0.8, alpha=0.7, color='#2196F3')
ax.fill_between(product_sales['sale_date'], product_sales['quantity_sold'], 
                alpha=0.15, color='#2196F3')
ax.set_title(f'Serie Temporal de Ventas — {product.name}', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Unidades Vendidas')
plt.tight_layout()
plt.show()

## 3. Ingeniería de Características (Feature Engineering)

La ingeniería de características transforma la serie temporal cruda de ventas en un conjunto de **40+ features** que capturan diferentes aspectos del comportamiento de la demanda.

### 3.1 Categorías de Features

| Categoría | Features | Propósito |
|-----------|----------|-----------|
| **Lags temporales** | `lag_1`, `lag_3`, `lag_7`, `lag_14`, `lag_21`, `lag_28`, `lag_365` | Capturar patrones de repetición diarios, semanales, mensuales y anuales |
| **Estadísticas móviles** | `rolling_mean_7/14/21`, `rolling_std_7/14` | Tendencia reciente y volatilidad |
| **Momentum** | `ewm_7` (suavizado exponencial), `trend_7d` (pendiente lineal) | Dirección del cambio reciente |
| **Calendario** | `day_of_week`, `month`, `is_weekend`, `is_payday`, `is_holiday` | Patrones estacionales del calendario venezolano |
| **Encoding cíclico** | `dow_sin/cos`, `month_sin/cos`, `dom_sin/cos` | Representación continua para que el modelo entienda que lunes y domingo están "cerca" |
| **Interacciones** | `weekend×mean7`, `payday×mean7`, `holiday×std7` | Efectos combinados no lineales |

In [ ]:
from core.ml.feature_engineering import build_features, FEATURE_COLUMNS

# Preparar datos para feature engineering
fe_input = product_sales[['sale_date', 'quantity_sold']].copy()

# Construir features
df_features = build_features(fe_input, shelf_life_days=product.shelf_life_days)

print(f'Features generadas: {len(FEATURE_COLUMNS)}')
print(f'Filas resultantes: {len(df_features)} (de {len(fe_input)} originales)')
print(f'\nLista completa de features:')
for i, col in enumerate(FEATURE_COLUMNS, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Visualizar features de lag
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

lag_features = ['lag_1', 'lag_7', 'lag_14', 'lag_28']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for ax, lag, color in zip(axes.flat, lag_features, colors):
    ax.scatter(df_features[lag], df_features['quantity_sold'], 
              alpha=0.3, s=10, color=color)
    ax.set_xlabel(f'{lag} (demanda hace {lag.split("_")[1]} día(s))')
    ax.set_ylabel('Demanda actual')
    ax.set_title(f'Autocorrelación: {lag}', fontweight='bold')

plt.suptitle(f'Relación entre Lags y Demanda — {product.name}', 
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar encoding cíclico
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Día de la semana (cíclico)
axes[0].scatter(df_features['dow_sin'], df_features['dow_cos'], 
               c=df_features['day_of_week'], cmap='viridis', s=5, alpha=0.5)
axes[0].set_title('Encoding Cíclico: Día de la Semana', fontweight='bold')
axes[0].set_xlabel('sin(día)')
axes[0].set_ylabel('cos(día)')

# Mes (cíclico)
axes[1].scatter(df_features['month_sin'], df_features['month_cos'],
               c=df_features['month'], cmap='coolwarm', s=5, alpha=0.5)
axes[1].set_title('Encoding Cíclico: Mes', fontweight='bold')
axes[1].set_xlabel('sin(mes)')
axes[1].set_ylabel('cos(mes)')

# Venta promedio por día de la semana
day_names = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
daily_avg = df_features.groupby('day_of_week')['quantity_sold'].mean()
axes[2].bar(range(7), daily_avg.values, color=['#4ECDC4' if i < 5 else '#FF6B6B' for i in range(7)])
axes[2].set_xticks(range(7))
axes[2].set_xticklabels(day_names)
axes[2].set_title('Demanda Promedio por Día', fontweight='bold')
axes[2].set_ylabel('Unidades')

plt.tight_layout()
plt.show()

## 4. Control de Outliers — Winsorización Adaptativa

La **winsorización** es una técnica que recorta valores extremos sin eliminar filas. Esto es esencial porque un pedido grande para una fiesta (ej: 15 tortas cuando lo normal son 2) distorsionaría los lags y las estadísticas móviles.

### Algoritmo

1. Calcular Q1, Q3 e IQR (Rango Intercuartílico)
2. **Si IQR > 0**: fence superior = Q3 + 1.5 × IQR
3. **Si IQR = 0** (producto de bajo volumen): fence superior = percentil 99
4. Recortar valores que excedan el fence (no se eliminan filas)

In [ ]:
from core.ml.feature_engineering import _winsorize

qty = product_sales['quantity_sold'].copy()
qty_winsorized = _winsorize(qty)

# Comparación antes/después
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(qty.values, vert=True)
axes[0].set_title('ANTES de Winsorización', fontweight='bold')
axes[0].set_ylabel('Unidades vendidas')

axes[1].boxplot(qty_winsorized.values, vert=True)
axes[1].set_title('DESPUÉS de Winsorización', fontweight='bold')
axes[1].set_ylabel('Unidades vendidas')

n_changed = int((qty != qty_winsorized).sum())
plt.suptitle(f'Efecto de la Winsorización — {product.name} ({n_changed} valores recortados)', 
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

q1, q3 = qty.quantile(0.25), qty.quantile(0.75)
iqr = q3 - q1
print(f'Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}')
print(f'Fence inferior = {max(0, q1 - 1.5*iqr):.2f}')
print(f'Fence superior = {q3 + 1.5*iqr:.2f}')
print(f'Valores recortados: {n_changed}')

## 5. Entrenamiento del Modelo

### 5.1 Validación Cruzada Temporal (TimeSeriesSplit)

A diferencia de la validación cruzada convencional (K-Fold), en series temporales **no podemos mezclar datos futuros con pasados**. Se usa `TimeSeriesSplit` que respeta el orden cronológico:

```
Fold 1: [████ TRAIN ████][▓▓ TEST ▓▓]
Fold 2: [████████ TRAIN ████████][▓▓ TEST ▓▓]
Fold 3: [████████████ TRAIN ████████████][▓▓ TEST ▓▓]
```

### 5.2 Estrategia de Entrenamiento por Tiers

El sistema selecciona automáticamente el pipeline según la cantidad de datos:

| Tier | Datos | Pipeline |
|------|-------|----------|
| **Alto** | ≥ 50 días | RandomizedSearchCV + competencia RF vs LightGBM |
| **Medio** | 21-49 días | RF con hiperparámetros conservadores fijos |
| **Lite** | 7-20 días | RF simplificado (50 árboles, profundidad 5) |
| **Fallback** | < 7 días | Media Móvil Exponencial (EWM) |

In [ ]:
from core.ml.model_trainer import train_product_model

# Preparar datos de entrada
train_input = product_sales[['sale_date', 'quantity_sold']].copy()
train_input['sale_date'] = train_input['sale_date'].dt.strftime('%Y-%m-%d')

print(f'Entrenando modelo para: {product.name} ({product.sku})')
print(f'Datos disponibles: {len(train_input)} registros')
print(f'Vida útil del producto: {product.shelf_life_days} días')
print()

# Entrenar
result = train_product_model(
    sales_df=train_input,
    product_id=product.id,
    sku=product.sku,
    shelf_life_days=product.shelf_life_days,
    n_cv_splits=3,
)

print('\n' + '='*50)
print('RESULTADO DEL ENTRENAMIENTO')
print('='*50)
print(f'Modelo: {result["version_tag"]}')
print(f'Filas de entrenamiento: {result["training_rows"]}')
print(f'MAPE: {result["mape_val"]:.4f} ({result["mape_val"]*100:.1f}%)')
print(f'RMSE: {result["rmse_val"]:.4f}')
print(f'MAE:  {result["mae_val"]:.4f}')

## 6. Evaluación del Modelo

### 6.1 Métricas de Error

| Métrica | Qué mide | Fórmula |
|---------|----------|----------|
| **MAPE** | Error porcentual promedio | $\frac{1}{n}\sum\frac{|y_i - \hat{y}_i|}{y_i}$ |
| **RMSE** | Error cuadrático (penaliza errores grandes) | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ |
| **MAE** | Error absoluto promedio (en unidades físicas) | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ |

> **Interpretación clave**: En un negocio de bajo volumen como una pastelería artesanal, el **MAE** es la métrica más relevante porque expresa el error en unidades de torta. Un MAE de 0.3 significa que el modelo se equivoca por menos de un tercio de torta al día.

In [ ]:
import joblib
from core.ml.feature_engineering import build_features, FEATURE_COLUMNS
from core.ml.pipeline import predict_with_intervals

# Cargar el modelo recién entrenado
pipeline = joblib.load(result['model_path'])

# Generar features para todo el dataset
fe_all = product_sales[['sale_date', 'quantity_sold']].copy()
df_all = build_features(fe_all, shelf_life_days=product.shelf_life_days)

# Predecir sobre el conjunto completo
X_all = df_all[FEATURE_COLUMNS]
y_actual = df_all['quantity_sold'].values
y_pred = pipeline.predict(X_all)
y_pred = np.maximum(0, y_pred)

# Feature Importance
model = pipeline.named_steps['model']
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    feat_imp = sorted(zip(FEATURE_COLUMNS, importances), key=lambda x: x[1], reverse=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = 15
    names = [f[0] for f in feat_imp[:top_n]][::-1]
    values = [f[1] for f in feat_imp[:top_n]][::-1]
    
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
    ax.barh(names, values, color=colors)
    ax.set_xlabel('Importancia Relativa')
    ax.set_title(f'Top {top_n} Features más Importantes — {product.name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Gráfico: Predicción vs Valor Real (serie temporal)
dates = df_all['sale_date'].values

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, y_actual, label='Demanda Real', linewidth=0.8, alpha=0.7, color='#2196F3')
ax.plot(dates, y_pred, label='Predicción RF', linewidth=0.8, alpha=0.7, color='#FF5722', linestyle='--')
ax.fill_between(dates, y_actual, y_pred, alpha=0.1, color='#FF5722')
ax.set_title(f'Predicción vs. Demanda Real — {product.name}', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Unidades')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Intervalos de Confianza

### ¿Cómo se calculan?

Random Forest es un **ensemble de N árboles de decisión**. Cada árbol da su propia predicción. Esto permite calcular intervalos de confianza de forma natural:

1. Cada uno de los 200 árboles predice un valor para el mismo día
2. Se toma el **percentil 5** como límite inferior (90% IC)
3. Se toma el **percentil 95** como límite superior
4. La **media** de todos los árboles es la predicción puntual

Esto es una ventaja sobre modelos como redes neuronales, que requieren técnicas adicionales (Monte Carlo Dropout, etc.) para estimar incertidumbre.

In [ ]:
# Generar predicciones con intervalos de confianza
intervals = predict_with_intervals(pipeline, X_all, confidence=0.90)

# Gráfico con bandas de confianza (últimos 60 días para mejor visualización)
n_show = 60
fig, ax = plt.subplots(figsize=(14, 5))

dates_show = dates[-n_show:]
actual_show = y_actual[-n_show:]
pred_show = intervals['predicted'].values[-n_show:]
lower_show = intervals['lower'].values[-n_show:]
upper_show = intervals['upper'].values[-n_show:]

ax.fill_between(dates_show, lower_show, upper_show, 
                alpha=0.2, color='#FF9800', label='Intervalo de Confianza 90%')
ax.plot(dates_show, actual_show, 'o-', markersize=3, linewidth=0.8, 
        color='#2196F3', label='Demanda Real')
ax.plot(dates_show, pred_show, '--', linewidth=1.2, 
        color='#FF5722', label='Predicción')

ax.set_title(f'Pronóstico con Intervalos de Confianza (90%) — {product.name}', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Unidades')
ax.legend()
plt.tight_layout()
plt.show()

# Cobertura del intervalo
within_bounds = np.sum((y_actual >= intervals['lower'].values) & 
                       (y_actual <= intervals['upper'].values))
coverage = within_bounds / len(y_actual) * 100
print(f'Cobertura del intervalo de confianza al 90%: {coverage:.1f}%')

## 8. Conexión con Investigación de Operaciones

Las predicciones del modelo alimentan dos algoritmos de optimización:

### 8.1 Modelo Newsvendor — ¿Cuántas tortas hornear?

$$Q^* = F^{-1}(CR) \quad \text{donde} \quad CR = \frac{P_v - C_u}{P_v}$$

### 8.2 Punto de Reorden (ROP) — ¿Cuándo pedir insumos?

$$ROP = \mu_d \times L + Z_\alpha \times \sqrt{L \times \sigma_d^2 + \mu_d^2 \times \sigma_L^2}$$

In [ ]:
from core.operations_research.newsvendor import calculate_critical_ratio, newsvendor_optimal_quantity

# Calcular ratio crítico
cr = calculate_critical_ratio(
    unit_cost=product.unit_cost,
    selling_price=product.selling_price
)

# Calcular cantidad óptima de producción
mu = float(intervals['predicted'].mean())
sigma = float(intervals['predicted'].std())

nv_result = newsvendor_optimal_quantity(
    mu_demand=mu,
    sigma_demand=sigma,
    critical_ratio=cr,
    min_order=product.min_order_qty
)

print(f'=== Análisis Newsvendor para {product.name} ===')
print(f'Precio de venta: ${product.selling_price:.2f}')
print(f'Costo unitario:  ${product.unit_cost:.2f}')
print(f'Ratio crítico:   {cr:.4f}')
print(f'Demanda estimada (μ): {mu:.2f} unidades/día')
print(f'Desviación (σ):      {sigma:.2f}')
print(f'\n→ Cantidad óptima a hornear (Q*): {nv_result["q_star_rounded"]} unidades/día')
print(f'→ Nivel de servicio: {nv_result["service_level_at_q"]*100:.1f}%')

In [ ]:
# Cerrar la sesión de base de datos
db.close()
print('Sesión de base de datos cerrada correctamente.')

---

## Referencias

- Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5-32.
- Porteus, E. L. (2002). *Foundations of Stochastic Inventory Theory*. Stanford University Press.
- Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. JMLR, 12, 2825-2830.
- Ke, G. et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*. NeurIPS.

---

*Notebook generado como parte del Trabajo Especial de Grado — Universidad de Oriente, Núcleo de Monagas, 2026*